# Lesson 07: PyTorch Experiment Tracking

This notebook covers:
- Why experiment tracking is essential
- Setting up TensorBoard with PyTorch
- Tracking metrics during training
- Comparing multiple experiments
- Running systematic experiments (different models, data amounts, epochs)
- Finding the best model configuration
- Making predictions with the best model

## Why Track Experiments?

When training models, you'll often ask:
- Which model performed best?
- How did different hyperparameters affect performance?
- Is my model overfitting or underfitting?
- How much training data do I actually need?

**Without tracking**: Chaos, forgetting what worked, repeating failed experiments

**With tracking**: Organized experiments, reproducible results, data-driven decisions

## What We'll Track:
- Training loss and accuracy
- Testing loss and accuracy
- Different data amounts (10% vs 20% of data)
- Different models (EfficientNet-B0 vs B2)
- Different training durations (5, 10, 20 epochs)

**Total experiments**: 2 datasets × 2 models × 3 epoch settings = 12 experiments!

## Setup and Imports

In [ ]:
# Import PyTorch libraries
import torch
import torchvision

# Check versions
print(torch.__version__)
print(torchvision.__version__)

In [ ]:
# Import standard libraries
import matplotlib.pyplot as plt 
import numpy as np
from torch import nn
from torchvision import transforms

import os
import zipfile
from pathlib import Path
import requests

In [ ]:
# Try to get torchinfo and going_modular
try:
    from torchinfo import summary
except:
    print("[INFO] Could not find torchinfo... installing it")
    !pip install -q torchinfo
    from torchinfo import summary

# Import our modular code from Lesson 05
try:
    from going_modular import data_setup, engine
except:
    print("[INFO] Could not find the going_modular scripts... downloading them now from Github")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular import data_setup, engine

In [ ]:
# Setup device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## Helper Functions

In [ ]:
def set_seeds(seed: int=42):
    """
    Sets random seeds for torch operations.
    
    This ensures reproducibility - same seeds = same results.
    Critical for comparing experiments fairly!

    Args:
        seed (int, optional): Random seed to set. Defaults to 42.
    """
    # Set the seed for general torch operations (CPU)
    torch.manual_seed(seed)
    # Set the seed for CUDA torch operations (GPU)
    torch.cuda.manual_seed(seed)

In [ ]:
def download_data(source: str,
                  destination: str,
                  remove_source: bool = True) -> Path:
    """
    Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after extraction.

    Returns:
        pathlib.Path to downloaded data directory.
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)

        # Download data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)

        # Unzip data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...")
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
            
    return image_path

In [ ]:
# Download the pizza, steak, sushi dataset
image_path = download_data(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
    destination="pizza_steak_sushi"
)
image_path

## Part 1: Create Transforms and DataLoaders

In [ ]:
# Setup directories
train_dir = image_path / "train"
test_dir = image_path / "test"

# Setup ImageNet normalization levels
# These are standard values used by most pretrained models
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],  # ImageNet mean (R, G, B)
    std=[0.229, 0.224, 0.225]    # ImageNet std (R, G, B)
)

# Create transform pipeline manually
manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to expected input size
    transforms.ToTensor(),          # Convert to tensor [0, 1]
    normalize                       # Normalize with ImageNet stats
])

## Part 2: Using Automatic Transforms from Pretrained Weights

In [ ]:
# Setup directories
train_dir = image_path / "train"
test_dir = image_path / "test"

# Setup pretrained weights
# .DEFAULT = best available pretrained weights
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

# Get transforms from weights (automatic way!)
# This ensures we use the EXACT same preprocessing as during pretraining
automatic_transforms = weights.transforms()
print(f"Automatically created transforms: {automatic_transforms}")

# Create dataloaders using automatic transforms
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=automatic_transforms,
    batch_size=32
)

## Part 3: Create Model with Pretrained Weights

In [ ]:
# Download the pretrained weights for EfficientNet_B0
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

# Setup the model with pretrained weights and send to device
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

# View the model architecture
model

In [ ]:
# Freeze all base layers (feature extraction approach)
# This prevents updating pretrained weights
for param in model.features.parameters():
    param.requires_grad = False

# Since we're creating a new layer with random weights,
# let's set the seeds for reproducibility
set_seeds()

# Update the classifier head to suit our problem
# Original: 1000 ImageNet classes → New: 3 classes (pizza, steak, sushi)
model.classifier = torch.nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),  # Regularization
    nn.Linear(
        in_features=1280,              # EfficientNet-B0 outputs 1280 features
        out_features=len(class_names)  # 3 classes for our dataset
    )
).to(device)

In [ ]:
# Get a summary of the model
# Notice: most parameters are NOT trainable (frozen)
summary(
    model,
    input_size=(32, 3, 224, 224),  # (batch_size, channels, height, width)
    verbose=0,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

## Part 4: Setup TensorBoard for Experiment Tracking

TensorBoard is a visualization toolkit that lets you:
- Track and visualize metrics (loss, accuracy)
- Compare multiple experiments
- Visualize model graphs
- View images and embeddings

In [ ]:
# Define loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Import TensorBoard SummaryWriter
try:
    from torch.utils.tensorboard import SummaryWriter
except:
    print(f"[INFO] Downloading tensorboard...")
    !pip install -q tensorboard
    from torch.utils.tensorboard import SummaryWriter

In [ ]:
# Create a writer with default settings
# This creates a 'runs' directory to store experiment logs
writer = SummaryWriter()

# The writer will save logs to: runs/[timestamp]/

## Part 5: Modified Training Function with TensorBoard Tracking

In [ ]:
from tqdm.auto import tqdm
from typing import Dict, List, Tuple
from going_modular.engine import train_step, test_step

def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    epochs: int,
    device: torch.device,
    writer: torch.utils.tensorboard.writer.SummaryWriter  # NEW: TensorBoard writer
) -> Dict[str, List[float]]:
    """
    Trains and tests a PyTorch model with TensorBoard tracking.

    Passes a target PyTorch model through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.
    
    NEW: Logs metrics to TensorBoard for visualization!

    Args:
        model: A PyTorch model to be trained and tested.
        train_dataloader: A DataLoader instance for the model to be trained on.
        test_dataloader: A DataLoader instance for the model to be tested on.
        optimizer: A PyTorch optimizer to help minimize the loss function.
        loss_fn: A PyTorch loss function to calculate loss on both datasets.
        epochs: An integer indicating how many epochs to train for.
        device: A target device to compute on (e.g. "cuda" or "cpu").
        writer: A SummaryWriter instance to log metrics to TensorBoard.

    Returns:
        A dictionary of training and testing loss as well as training and
        testing accuracy metrics.
    """
    # Create empty results dictionary
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        # Perform training step
        train_loss, train_acc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )
        
        # Perform testing step
        test_loss, test_acc = test_step(
            model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device
        )

        # Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        # NEW: Track experiments with TensorBoard
        # Log metrics to TensorBoard at each epoch
        if writer:
            # Add results to SummaryWriter (TensorBoard)
            writer.add_scalars(
                main_tag="Loss",
                tag_scalar_dict={"train_loss": train_loss, "test_loss": test_loss},
                global_step=epoch
            )
            writer.add_scalars(
                main_tag="Accuracy",
                tag_scalar_dict={"train_acc": train_acc, "test_acc": test_acc},
                global_step=epoch
            )

            # Close the writer when done
            # writer.close()  # Don't close here, we might use it again!

    # Return the filled results at the end of the epochs
    return results

In [ ]:
# Train model with TensorBoard tracking
# Note: Only training for 1 epoch as a test
set_seeds()

results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=1,
    device=device,
    writer=writer  # Pass the TensorBoard writer
)

## Viewing TensorBoard Results

To view TensorBoard in Jupyter/Colab:
```python
%load_ext tensorboard
%tensorboard --logdir runs
```

To view from command line:
```bash
tensorboard --logdir runs
```

Then open browser to: http://localhost:6006

In [ ]:
# Example code to run in Jupyter or Google Colab (uncomment to use)
# %load_ext tensorboard
# %tensorboard --logdir runs

## Part 6: Creating Organized Experiment Logging

Let's create a function to organize our TensorBoard logs better.

In [ ]:
def create_writer(
    experiment_name: str,
    model_name: str,
    extra: str=None
) -> torch.utils.tensorboard.writer.SummaryWriter():
    """
    Creates a torch.utils.tensorboard.writer.SummaryWriter() instance
    saving to a specific log_dir.

    log_dir is a combination of runs/timestamp/experiment_name/model_name/extra.

    Where timestamp is the current date in YYYY-MM-DD format.

    Args:
        experiment_name (str): Name of experiment.
        model_name (str): Name of model.
        extra (str, optional): Anything extra to add to the directory.

    Returns:
        torch.utils.tensorboard.writer.SummaryWriter(): Instance of a writer
        saving to log_dir.

    Example usage:
        # Create a writer saving to "runs/2022-06-04/data_10_percent/effnetb2/5_epochs/"
        writer = create_writer(experiment_name="data_10_percent",
                             model_name="effnetb2",
                             extra="5_epochs")
    """
    from datetime import datetime
    import os

    # Get current date in YYYY-MM-DD format
    timestamp = datetime.now().strftime("%Y-%m-%d")

    # Create log directory path
    # Format: runs/YYYY-MM-DD/experiment_name/model_name/extra
    if extra:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)

    # Create SummaryWriter with custom log_dir
    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")
    return SummaryWriter(log_dir=log_dir)

In [ ]:
# Example of using create_writer
example_writer = create_writer(
    experiment_name="data_10_percent",
    model_name="effnetb0",
    extra="5_epochs"
)

# This creates: runs/2024-XX-XX/data_10_percent/effnetb0/5_epochs/

## Part 7: Running Systematic Experiments

Now we'll run multiple experiments systematically:
- **2 data amounts**: 10% and 20% of training data
- **2 models**: EfficientNet-B0 and EfficientNet-B2
- **3 epoch settings**: 5, 10, and 20 epochs

**Total**: 2 × 2 × 3 = **12 experiments**

This helps us answer:
- Does more data help?
- Does a bigger model help?
- Does training longer help?
- What's the best combination?

In [ ]:
# Download 10% and 20% training data
data_10_percent_path = download_data(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
    destination="pizza_steak_sushi"
)

data_20_percent_path = download_data(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
    destination="pizza_steak_sushi_20_percent"
)

In [ ]:
# Setup training directory paths
train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"

# Use same test set for fair comparison
test_dir = data_10_percent_path / "test"

print(f"Training directory 10%: {train_dir_10_percent}")
print(f"Training directory 20%: {train_dir_20_percent}")
print(f"Testing directory: {test_dir}")

In [ ]:
# Create simple transform
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

simple_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

In [ ]:
BATCH_SIZE = 32

# Create 10% DataLoaders
train_dataloader_10_percent, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir_10_percent,
    test_dir=test_dir,
    transform=simple_transform,
    batch_size=BATCH_SIZE
)

# Create 20% DataLoaders
train_dataloader_20_percent, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir_20_percent,
    test_dir=test_dir,
    transform=simple_transform,
    batch_size=BATCH_SIZE
)

### Create Model Builder Functions

These functions create fresh model instances for each experiment.

In [ ]:
OUT_FEATURES = len(class_names)  # 3 classes

def create_effnetb0():
    """
    Creates an EfficientNet-B0 feature extractor model.
    
    Returns:
        EfficientNet-B0 with frozen features and custom classifier.
    """
    # Get pretrained weights
    weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
    model = torchvision.models.efficientnet_b0(weights=weights).to(device)

    # Freeze feature layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Set seeds before creating new layers
    set_seeds()

    # Replace classifier
    model.classifier = torch.nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features=1280, out_features=OUT_FEATURES)
    ).to(device)

    # Give model a name
    model.name = "effnetb0"
    return model


def create_effnetb2():
    """
    Creates an EfficientNet-B2 feature extractor model.
    
    B2 is larger than B0: more parameters, better accuracy (usually).
    
    Returns:
        EfficientNet-B2 with frozen features and custom classifier.
    """
    # Get pretrained weights
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    model = torchvision.models.efficientnet_b2(weights=weights).to(device)

    # Freeze feature layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Set seeds
    set_seeds()

    # Replace classifier
    # Note: B2 outputs 1408 features (vs 1280 for B0)
    model.classifier = torch.nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),  # Slightly higher dropout for larger model
        nn.Linear(in_features=1408, out_features=OUT_FEATURES)
    ).to(device)

    # Give model a name
    model.name = "effnetb2"
    return model

### Run All Experiments

This loop runs all 12 experiments and saves:
- TensorBoard logs for visualization
- Model checkpoints for the best models

In [ ]:
%%time
# Import utilities
from going_modular.utils import save_model
from datetime import datetime

# Define experiment grid
num_epochs = [5, 10, 20]  # Different training durations
models = ["effnetb0", "effnetb2"]  # Different model architectures
train_dataloaders = {
    "data_10_percent": train_dataloader_10_percent,
    "data_20_percent": train_dataloader_20_percent
}

# Track experiment numbers
experiment_number = 0

# Loop through each DataLoader (data amount)
for dataloader_name, train_dataloader in train_dataloaders.items():
    
    # Loop through each number of epochs
    for epochs in num_epochs:
        
        # Loop through each model
        for model_name in models:
            
            experiment_number += 1
            print(f"\n[INFO] Experiment number: {experiment_number}")
            print(f"[INFO] Model: {model_name}")
            print(f"[INFO] DataLoader: {dataloader_name}")
            print(f"[INFO] Number of epochs: {epochs}")

            # Create model based on name
            if model_name == "effnetb0":
                model = create_effnetb0()
            else:
                model = create_effnetb2()

            # Create loss and optimizer
            loss_fn = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

            # Create TensorBoard writer
            # Log directory will be: runs/YYYY-MM-DD/data_X_percent/model_name/X_epochs/
            writer = create_writer(
                experiment_name=dataloader_name,
                model_name=model_name,
                extra=f"{epochs}_epochs"
            )

            # Train model
            print(f"[INFO] Training {model_name} for {epochs} epochs...")
            train(
                model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=epochs,
                device=device,
                writer=writer
            )

            # Save model
            save_model(
                model=model,
                target_dir="models",
                model_name=f"07_{model_name}_{dataloader_name}_{epochs}_epochs.pth"
            )
            print("-" * 50)

## Viewing All Experiments in TensorBoard

After running all experiments, view them in TensorBoard:

```python
%tensorboard --logdir runs
```

Or from command line:
```bash
tensorboard --logdir runs
```

You can now compare:
- Loss curves across all experiments
- Accuracy trends
- Which combination performed best
- Overfitting patterns (gap between train and test)

## Part 8: Loading and Using the Best Model

In [ ]:
# After viewing TensorBoard, identify the best model
# For example: effnetb2_data_20_percent_10_epochs

# Setup the best model filepath
best_model_path = "models/07_effnetb2_data_20_percent_10_epochs.pth"

# Instantiate a new model (same architecture)
best_model = create_effnetb2()

# Load the saved weights
best_model.load_state_dict(torch.load(best_model_path))

print(f"[INFO] Loaded best model from: {best_model_path}")

In [ ]:
# Check model file size
from pathlib import Path

model_size = Path(best_model_path).stat().st_size // (1024*1024)  # Convert to MB
print(f"Model size: {model_size} MB")

## Making Predictions with Best Model

In [ ]:
# Make predictions on random test images
import random

num_images_to_plot = 3
test_image_path_list = list(Path(test_dir).glob("*/*.jpg"))
test_image_path_sample = random.sample(
    population=test_image_path_list,
    k=num_images_to_plot
)

# Make predictions and plot
for image_path in test_image_path_sample:
    pred_and_plot_image(
        model=best_model,
        image_path=image_path,
        class_names=class_names,
        transform=simple_transform
    )